In [1]:
# ============================================================
# PPE SAFETY SYSTEM WITH BYTE TRACKING
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

from ultralytics import YOLO

import cv2
import time

from pathlib import Path

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path("../")

MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "ppe_yolo11m_best.pt"

INPUT_VIDEO = PROJECT_ROOT / "videos" / "input" / "input.mp4"

OUTPUT_VIDEO = PROJECT_ROOT / "videos" / "output" / "02_tracking_demo.mp4"

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(str(MODEL_PATH))

print("=" * 60)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 60)

# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(str(INPUT_VIDEO))

if not cap.isOpened():
    raise ValueError(f"Cannot open video: {INPUT_VIDEO}")

# ============================================================
# VIDEO PROPERTIES
# ============================================================

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Resolution : {frame_width} x {frame_height}")
print(f"FPS        : {fps}")

# ============================================================
# OUTPUT VIDEO WRITER
# ============================================================

OUTPUT_VIDEO.parent.mkdir(
    parents=True,
    exist_ok=True
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    fps,
    (frame_width, frame_height)
)

# ============================================================
# PERFORMANCE VARIABLES
# ============================================================

prev_time = 0
frame_count = 0

# ============================================================
# TRACK HISTORY
# ============================================================

track_history = {}

# ============================================================
# CLASS NAMES
# ============================================================

CLASS_NAMES = model.names

# ============================================================
# INFERENCE LOOP
# ============================================================

print("\nStarting tracked video inference...\n")

while True:

    success, frame = cap.read()

    if not success:
        print("\nVideo processing completed.")
        break

    # --------------------------------------------------------
    # YOLO + BYTE TRACKING
    # --------------------------------------------------------

    results = model.track(

        source=frame,

        persist=True,

        tracker="bytetrack.yaml",

        conf=0.50,

        imgsz=960,

        verbose=False,

        device=0
    )

    result = results[0]

    annotated_frame = frame.copy()

    # --------------------------------------------------------
    # PROCESS DETECTIONS
    # --------------------------------------------------------

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()

        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        confidences = result.boxes.conf.cpu().numpy()

        track_ids = result.boxes.id.cpu().numpy().astype(int)

        # ----------------------------------------------------
        # LOOP THROUGH DETECTIONS
        # ----------------------------------------------------

        for box, class_id, conf, track_id in zip(
            boxes,
            class_ids,
            confidences,
            track_ids
        ):

            x1, y1, x2, y2 = map(int, box)

            class_name = CLASS_NAMES[class_id]

            # ------------------------------------------------
            # COLOR SETTINGS
            # ------------------------------------------------

            if class_name == "Person":
                color = (0, 255, 0)

            elif class_name in ["Hardhat", "Safety Vest"]:
                color = (255, 255, 0)

            else:
                color = (0, 165, 255)

            # ------------------------------------------------
            # DRAW BOUNDING BOX
            # ------------------------------------------------

            cv2.rectangle(
                annotated_frame,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            # ------------------------------------------------
            # LABEL
            # ------------------------------------------------

            label = (
                f"ID {track_id} | "
                f"{class_name} | "
                f"{conf:.2f}"
            )

            cv2.putText(
                annotated_frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

            # ------------------------------------------------
            # TRACK CENTER POINT
            # ------------------------------------------------

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            # ------------------------------------------------
            # STORE TRACK HISTORY
            # ------------------------------------------------

            if track_id not in track_history:
                track_history[track_id] = []

            track_history[track_id].append(
                (center_x, center_y)
            )

            # Keep limited trajectory history
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

            # ------------------------------------------------
            # DRAW TRAJECTORY
            # ------------------------------------------------

            points = track_history[track_id]

            for i in range(1, len(points)):

                cv2.line(
                    annotated_frame,
                    points[i - 1],
                    points[i],
                    color,
                    2
                )

    # --------------------------------------------------------
    # FPS CALCULATION
    # --------------------------------------------------------

    current_time = time.time()

    fps_value = 1 / (current_time - prev_time)

    prev_time = current_time

    # --------------------------------------------------------
    # DISPLAY FPS
    # --------------------------------------------------------

    cv2.putText(
        annotated_frame,
        f"FPS: {fps_value:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # --------------------------------------------------------
    # DISPLAY FRAME COUNT
    # --------------------------------------------------------

    frame_count += 1

    cv2.putText(
        annotated_frame,
        f"Frame: {frame_count}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),
        2
    )

    # --------------------------------------------------------
    # SHOW VIDEO
    # --------------------------------------------------------

    cv2.imshow(
        "PPE Safety Tracking System",
        annotated_frame
    )

    # --------------------------------------------------------
    # SAVE FRAME
    # --------------------------------------------------------

    out.write(annotated_frame)

    # --------------------------------------------------------
    # EXIT KEY
    # --------------------------------------------------------

    key = cv2.waitKey(1)

    if key == ord("q"):
        print("\nInference stopped by user.")
        break

# ============================================================
# RELEASE RESOURCES
# ============================================================

cap.release()

out.release()

cv2.destroyAllWindows()

# ============================================================
# DONE
# ============================================================

print("=" * 60)
print("TRACKED VIDEO SAVED SUCCESSFULLY")
print("=" * 60)

print(f"\nSaved Output:\n{OUTPUT_VIDEO}")

MODEL LOADED SUCCESSFULLY
Resolution : 1920 x 1080
FPS        : 30

Starting tracked video inference...


Video processing completed.
TRACKED VIDEO SAVED SUCCESSFULLY

Saved Output:
C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\videos\output\tracked_output.mp4
